# Building Maintenance and Manual Intervention Notebook

This notebook helps with manual intervention for building details that couldn't be automatically matched.

## Overview
- Each pipeline (centaline_oir, midland_ici, midland_res) ends with `xxx_base`
- Inside that, there's building-to-transaction mapping
- This notebook identifies high-frequency buildings that need manual detail entry
- Creates Excel templates for manual data entry
- Loads manually entered data back into the system

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Define data paths
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "01_raw"
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
PRIMARY_DIR = DATA_DIR / "03_primary"
REPORTING_DIR = DATA_DIR / "08_reporting"

## Step 1: Analyze Buildings Needing Manual Intervention

Let's examine each pipeline's base data to find buildings that couldn't be matched automatically.

In [ ]:
def analyze_unmatched_buildings():
    """Analyze buildings that need manual intervention across all pipelines"""
    
    results = {}
    
    # Define pipeline configurations
    pipelines = {
        'centaline_oir': {
            'base_file': INTERMEDIATE_DIR / 'centaline_oir_base.parquet',
            'details_file': INTERMEDIATE_DIR / 'centanet_oir_details.parquet',
            'building_cols': ['propertyNameCn', 'propertyNameEn'],
            'merge_status_col': '_merge_status'
        },
        'midland_ici': {
            'base_file': INTERMEDIATE_DIR / 'midland_ici_base.parquet',
            'details_file': INTERMEDIATE_DIR / 'midland_ici_building_details.parquet',
            'building_cols': ['eng_name', 'chi_name'],
            'merge_status_col': '_merge_status'
        },
        'midland_res': {
            'base_file': INTERMEDIATE_DIR / 'midland_res_base.parquet',
            'details_file': INTERMEDIATE_DIR / 'midland_res_estates.parquet',
            'building_cols': ['estate', 'building'],
            'merge_status_col': '_merge_status'
        }
    }
    
    for pipeline_name, config in pipelines.items():
        logger.info(f"Analyzing {pipeline_name}...")
        
        try:
            # Load base data
            base_df = pd.read_parquet(config['base_file'])
            logger.info(f"Loaded {len(base_df)} records from {pipeline_name}_base")
            
            # Find unmatched buildings
            if config['merge_status_col'] in base_df.columns:
                unmatched = base_df[base_df[config['merge_status_col']] == 'no_building_match']
                logger.info(f"Found {len(unmatched)} unmatched records")
                
                # Get frequency of unmatched buildings
                building_freq = {}
                for col in config['building_cols']:
                    if col in unmatched.columns:
                        freq = unmatched[col].value_counts()
                        building_freq[col] = freq
                        
                        # Show top high-frequency buildings
                        high_freq = freq[freq >= 5]  # Buildings appearing 5+ times
                        if len(high_freq) > 0:
                            logger.info(f"High-frequency unmatched {col}: {len(high_freq)} buildings")
                            logger.info(f"Top 5: {high_freq.head(5).to_dict()}")
                
                results[pipeline_name] = {
                    'total_unmatched': len(unmatched),
                    'building_freq': building_freq,
                    'high_freq_buildings': {
                        col: freq[freq >= 5].index.tolist() 
                        for col, freq in building_freq.items()
                    }
                }
            else:
                logger.warning(f"No {config['merge_status_col']} column found in {pipeline_name}_base")
                
        except Exception as e:
            logger.error(f"Error analyzing {pipeline_name}: {e}")
            results[pipeline_name] = {'error': str(e)}
    
    return results

# Run analysis
analysis_results = analyze_unmatched_buildings()

# Display summary
for pipeline, result in analysis_results.items():
    print(f"\n=== {pipeline.upper()} ===")
    if 'error' in result:
        print(f"Error: {result['error']}")
    else:
        print(f"Total unmatched records: {result['total_unmatched']}")
        for col, buildings in result['high_freq_buildings'].items():
            print(f"High-frequency buildings in {col}: {len(buildings)}")

## Step 2: Create Excel Template for Manual Data Entry

Now let's create Excel templates for the high-frequency buildings that need manual intervention.

In [ ]:
def get_building_details_columns(pipeline_name):
    """Get the expected columns for building details files"""
    
    column_schemas = {
        'centaline_oir': [
            'propertyNameCn', 'propertyNameEn', 'propertyUsageDisplayName', 
            'floor', 'unit', 'transactionArea', 'sourceDisplayName', 'price',
            'avgPrice', 'grade', 'districtNameEn', 'zoneEn', 'completion_year',
            'age', 'source_url', 'full_address', 'management_company', 'developers',
            'carpark', 'matched_building_name', 'match_score', 'property_type',
            'Datasource', 'id'
        ],
        'midland_ici': [
            'pis_bldg_id', 'eng_name', 'chi_name', 'ics_type', 'dist_name_en',
            'floor', 'flat', 'area', 'price', 'price_per_feet', 'street_name_zh',
            'street_name_en', 'streetno', 'area1', 'area_desc1', 'area2', 'area_desc2',
            'area3', 'area_desc3', 'area4', 'area_desc4', 'URL', 'Completion',
            'age', 'upload_source', 'No. of Floors', 'Management Fee (Approx. per sq. ft.)',
            'Floor Remark', 'Grade', 'Management Company', 'Datasource'
        ],
        'midland_res': [
            'estate_id', 'building_id', 'estate', 'building', 'first_op_date',
            'building_first_op_date', 'update_date', 'total_unit_count', 'total_block_count',
            'primary_school_net', 'developer_name', 'location_lat', 'location_lon',
            'parent_estate_id', 'parent_estate_name', 'housing_type', 'amenities',
            'sm_district', 'int_district_id_estate', 'int_district', 'int_sm_district_id_estate',
            'int_sm_district', 'sm_district_name', 'region_name_estate', 'subregion_name',
            'district_name', 'int_district_id_trans', 'int_sm_district_id_trans',
            'int_sm_district_name', 'tags', 'Datasource'
        ]
    }
    
    return column_schemas.get(pipeline_name, [])

def create_manual_entry_template(pipeline_name, high_freq_buildings, output_dir=None):
    """Create Excel template for manual building details entry"""
    
    if output_dir is None:
        output_dir = Path("data/03_primary")
    
    logger.info(f"Creating manual entry template for {pipeline_name}")
    
    # Get building details columns
    columns = get_building_details_columns(pipeline_name)
    
    if not columns:
        logger.error(f"No column schema found for {pipeline_name}")
        return None
    
    # Determine building identifier columns
    building_configs = {
        'centaline_oir': ['propertyNameCn', 'propertyNameEn'],
        'midland_ici': ['eng_name', 'chi_name'],
        'midland_res': ['estate', 'building']
    }
    
    building_cols = building_configs.get(pipeline_name, [])
    
    # Create template DataFrame
    template_data = []
    
    # Add rows for each high-frequency building
    for building_col, buildings in high_freq_buildings.items():
        if building_col in building_cols:
            for building_name in buildings:
                row = {col: '' for col in columns}
                # Set the building identifier
                row[building_col] = building_name
                row['Datasource'] = pipeline_name.split('_')[0].title()  # Centaline or Midland
                template_data.append(row)
    
    if not template_data:
        logger.warning(f"No high-frequency buildings found for {pipeline_name}")
        return None
    
    template_df = pd.DataFrame(template_data)
    
    # Save to Excel
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{pipeline_name}_manual_building_details_{timestamp}.xlsx"
    output_path = output_dir / filename
    
    # Create Excel with formatting instructions
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        template_df.to_excel(writer, sheet_name='Manual_Entry', index=False)
        
        # Add instruction sheet
        instructions_df = pd.DataFrame({
            'Instructions': [
                f'Manual Building Details Entry for {pipeline_name.upper()}',
                '',
                'PURPOSE:',
                f'Enter building details for {len(template_df)} high-frequency buildings that could not be automatically matched.',
                '',
                'INSTRUCTIONS:',
                '1. Fill in the details for each building in the Manual_Entry sheet',
                '2. Do not modify the building identifier columns',
                '3. Save this file and run the notebook again to load the data',
                '',
                'REQUIRED FIELDS:',
                '- Address/location information',
                '- Building specifications (floors, units, etc.)',
                '- Developer and management company info',
                '- Any other available building details',
                '',
                f'Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
                f'Pipeline: {pipeline_name}',
                f'Buildings to process: {len(template_df)}'
            ]
        })
        instructions_df.to_excel(writer, sheet_name='Instructions', index=False)
    
    logger.info(f"Created manual entry template: {output_path}")
    logger.info(f"Template contains {len(template_df)} buildings requiring manual entry")
    
    return output_path

# Create templates for all pipelines with high-frequency unmatched buildings
templates_created = []

for pipeline_name, result in analysis_results.items():
    if 'high_freq_buildings' in result:
        template_path = create_manual_entry_template(pipeline_name, result['high_freq_buildings'])
        if template_path:
            templates_created.append((pipeline_name, template_path))

print("\n=== TEMPLATES CREATED ===")
for pipeline, path in templates_created:
    print(f"{pipeline}: {path}")

## Step 3: Load Manually Entered Data Back into System

After you manually fill in the Excel templates, run this section to load the data back into the building details files.

In [ ]:
def load_manual_building_details(manual_excel_path, pipeline_name):
    """Load manually entered building details back into the system"""
    
    logger.info(f"Loading manual building details for {pipeline_name} from {manual_excel_path}")
    
    # Load manual data
    try:
        manual_df = pd.read_excel(manual_excel_path, sheet_name='Manual_Entry')
        logger.info(f"Loaded {len(manual_df)} manual entries")
    except Exception as e:
        logger.error(f"Error loading manual data: {e}")
        return False
    
    # Filter out empty rows (buildings without manual data)
    # Consider a row complete if it has at least address/location info
    required_fields = {
        'centaline_oir': ['full_address'],
        'midland_ici': ['street_name_en', 'street_name_zh'],
        'midland_res': ['developer_name']
    }
    
    required = required_fields.get(pipeline_name, [])
    if required:
        # Keep rows that have at least one required field filled
        has_data = manual_df[required].notna().any(axis=1) & (manual_df[required] != '').any(axis=1)
        manual_df = manual_df[has_data]
        logger.info(f"After filtering empty rows: {len(manual_df)} entries with data")
    
    if len(manual_df) == 0:
        logger.warning("No valid manual entries found")
        return False
    
    # Load existing building details
    details_files = {
        'centaline_oir': INTERMEDIATE_DIR / 'centanet_oir_details.parquet',
        'midland_ici': INTERMEDIATE_DIR / 'midland_ici_building_details.parquet',
        'midland_res': INTERMEDIATE_DIR / 'midland_res_estates.parquet'
    }
    
    details_file = details_files.get(pipeline_name)
    if not details_file.exists():
        logger.info(f"Creating new building details file: {details_file}")
        existing_df = pd.DataFrame()
    else:
        try:
            existing_df = pd.read_parquet(details_file)
            logger.info(f"Loaded {len(existing_df)} existing building details")
        except Exception as e:
            logger.error(f"Error loading existing details: {e}")
            return False
    
    # Add manual entries to existing data
    # Remove duplicates based on building identifiers
    building_id_cols = {
        'centaline_oir': ['propertyNameCn', 'propertyNameEn'],
        'midland_ici': ['eng_name', 'chi_name'],
        'midland_res': ['estate', 'building']
    }
    
    id_cols = building_id_cols.get(pipeline_name, [])
    
    # Create identifier for deduplication
    if id_cols:
        manual_df['_building_id'] = manual_df[id_cols].fillna('').agg(' | '.join, axis=1)
        if len(existing_df) > 0:
            existing_df['_building_id'] = existing_df[id_cols].fillna('').agg(' | '.join, axis=1)
            
            # Remove existing entries that we're updating
            existing_df = existing_df[~existing_df['_building_id'].isin(manual_df['_building_id'])]
            existing_df = existing_df.drop('_building_id', axis=1)
        
        manual_df = manual_df.drop('_building_id', axis=1)
    
    # Combine datasets
    updated_df = pd.concat([existing_df, manual_df], ignore_index=True)
    
    # Save updated building details
    updated_df.to_parquet(details_file, index=False)
    
    logger.info(f"Successfully updated {details_file} with {len(manual_df)} manual entries")
    logger.info(f"Total building details now: {len(updated_df)}")
    
    # Create backup of the manual file
    backup_path = str(manual_excel_path).replace('.xlsx', f'_processed_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx')
    manual_df.to_excel(backup_path, index=False)
    logger.info(f"Created backup of processed manual data: {backup_path}")
    
    return True

# Example usage (uncomment and modify paths as needed):
# load_manual_building_details(
#     manual_excel_path="data/03_primary/centaline_oir_manual_building_details_20241019_143000.xlsx",
#     pipeline_name="centaline_oir"
# )

print("Manual data loading function is ready.")
print("Uncomment the example above and modify the path to load your manually entered data.")

## Step 4: Verify Improvements After Manual Intervention

After loading the manual data, run a new Kedro pipeline to see the improvement in building matching.

In [ ]:
def verify_improvements():
    """Verify the improvements after manual intervention"""
    
    print("=== BUILDING MATCHING IMPROVEMENTS ===\n")
    
    pipelines = ['centaline_oir', 'midland_ici', 'midland_res']
    
    for pipeline in pipelines:
        base_file = INTERMEDIATE_DIR / f'{pipeline}_base.parquet'
        
        if base_file.exists():
            try:
                df = pd.read_parquet(base_file)
                
                if '_merge_status' in df.columns:
                    total_records = len(df)
                    matched = len(df[df['_merge_status'] != 'no_building_match'])
                    unmatched = len(df[df['_merge_status'] == 'no_building_match'])
                    match_rate = (matched / total_records * 100) if total_records > 0 else 0
                    
                    print(f"{pipeline.upper()}:")
                    print(f"  Total records: {total_records:,}")
                    print(f"  Matched: {matched:,}")
                    print(f"  Unmatched: {unmatched:,}")
                    print(f"  Match rate: {match_rate:.1f}%")
                    print()
                else:
                    print(f"{pipeline.upper()}: No _merge_status column found")
                    
            except Exception as e:
                print(f"{pipeline.upper()}: Error loading data - {e}")
        else:
            print(f"{pipeline.upper()}: Base file not found")

# Run verification
verify_improvements()

## Usage Instructions

1. **Run Step 1**: Analyze which buildings need manual intervention
2. **Run Step 2**: Create Excel templates for manual data entry
3. **Manual Step**: Fill in the Excel templates with building details
4. **Run Step 3**: Load the manually entered data back into the system
5. **Run Step 4**: Verify the improvements in building matching
6. **Final Step**: Run `kedro run` to process the updated data through the full pipeline

## Notes

- Only high-frequency buildings (appearing 5+ times) are included in manual templates
- Templates are saved in `data/03_primary/` with timestamps
- Manual data is merged with existing building details
- Duplicates are automatically handled during the merge process